In [1]:
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_squared_error
import numpy as np
import itertools
import os

# Load and prepare your data as before
csv_path = 'all_temperature_cleaned.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
df['datetime'] = pd.to_datetime(df['timestamp'] + ' ' + df['time'], format='%Y-%m-%d %H:%M')

region = 'Rakhiyal'  # Example region; loop over regions as needed

train_df = df[(df['datetime'].dt.year >= 2019) & (df['datetime'].dt.year <= 2023)].copy()
test_df = df[df['datetime'].dt.year == 2024].copy()

train_prophet = train_df[['datetime', region]].rename(columns={'datetime': 'ds', region: 'y'})
test_prophet = test_df[['datetime', region]].rename(columns={'datetime': 'ds', region: 'y'})

# Define parameter grid
changepoint_prior_scale_values = [0.001, 0.01, 0.1, 0.5]
seasonality_prior_scale_values = [0.01, 0.1, 1.0, 10.0]
seasonality_modes = ['additive', 'multiplicative']

param_grid = list(itertools.product(changepoint_prior_scale_values, seasonality_prior_scale_values, seasonality_modes))

results = []

for cps, sps, smode in param_grid:
    model = Prophet(changepoint_prior_scale=cps,
                    seasonality_prior_scale=sps,
                    seasonality_mode=smode,
                    daily_seasonality=True,
                    yearly_seasonality=True,
                    weekly_seasonality=True)
    model.fit(train_prophet)

    future = test_prophet[['ds']].copy()
    forecast = model.predict(future)

    y_true = test_prophet['y'].values
    y_pred = forecast['yhat'].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    results.append({
        'changepoint_prior_scale': cps,
        'seasonality_prior_scale': sps,
        'seasonality_mode': smode,
        'rmse_2024': rmse
    })

# Convert results to DataFrame and find best parameters
results_df = pd.DataFrame(results)
best_row = results_df.loc[results_df['rmse_2024'].idxmin()]

print("Best parameters:")
print(best_row)

# Optionally save results
results_df.to_csv('prophet_param_tuning_results.csv', index=False)


C:\Users\PARIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
12:02:37 - cmdstanpy - INFO - Chain [1] start processing
12:02:41 - cmdstanpy - INFO - Chain [1] done processing
12:02:47 - cmdstanpy - INFO - Chain [1] start processing
12:02:54 - cmdstanpy - INFO - Chain [1] done processing
12:02:59 - cmdstanpy - INFO - Chain [1] start processing
12:03:05 - cmdstanpy - INFO - Chain [1] done processing
12:03:10 - cmdstanpy - INFO - Chain [1] start processing
12:03:17 - cmdstanpy - INFO - Chain [1] done processing
12:03:23 - cmdstanpy - INFO - Chain [1] start processing
12:03:31 - cmdstanpy - INFO - Chain [1] done processing
12:03:36 - cmdstanpy - INFO - Chain [1] start processing
12:03:42 - cmdstanpy - INFO - Chain [1] done processing
12:03:47 - cmdstanpy - INFO - Chain [1] star

Best parameters:
changepoint_prior_scale        0.01
seasonality_prior_scale         0.1
seasonality_mode           additive
rmse_2024                  2.157535
Name: 10, dtype: object
